# 19 · Shrinkage sensitivity of scope gaps — Gate 20

**Purpose.** Evaluate whether the main female-minus-male FPT gap, its residential and mobility-kernel components, and the paired regional-minus-district contrast depend materially on the row-shrinkage parameter.

The complete point-estimate pipeline is reconstructed for $\tau\in\{0,10,20,50\}$ using the same validated scopes and `core_50` target sets. The $\tau=20$ estimates must reproduce Gates 11–13. This notebook is a point-estimate sensitivity analysis; simultaneous confidence intervals remain those of the primary $\tau=20$ specification.

No monotonic relationship in $\tau$ is assumed or required.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn geopandas networkx


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, re, shutil, unicodedata, zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate20"
OUT = ROOT / "outputs/bogota_em2023/gate20"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (RAW, INTERIM, OUT): directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip"
ZONING_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip"
PURPOSES = ("employment", "education", "health")
EFFECTS = ("female_minus_male_total", "residential_contribution", "kernel_contribution")
TAU_VALUES = (0.0, 10.0, 20.0, 50.0)
PRIMARY_TAU = 20.0

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

G4 = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
G6 = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))
G11 = Path(os.environ.get("GATE11_DIR_OVERRIDE", DRIVE_BASE / "results/gate11"))
G12 = Path(os.environ.get("GATE12_DIR_OVERRIDE", DRIVE_BASE / "results/gate12"))
paths = {
    "region_edges": G4 / "transition_edges_utam.parquet",
    "region_targets": G6 / "purpose_target_sets.csv",
    "region_point": G11 / "final_female_male_point_estimates.csv",
    "city_targets": G12 / "bogota_dc_internal_core50_targets.csv",
    "city_point": G12 / "bogota_dc_internal_female_male_point_estimates.csv",
    "gate11": G11 / "gate_11.json", "gate12": G12 / "gate_12.json",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing: raise FileNotFoundError("Faltan insumos validados: " + ", ".join(missing))
gate11 = json.loads(paths["gate11"].read_text(encoding="utf-8"))
gate12 = json.loads(paths["gate12"].read_text(encoding="utf-8"))
if not gate11.get("gate", {}).get("gate_11_passed", False): raise RuntimeError("Gate 11 no aprobada")
if not gate12.get("gate", {}).get("gate_12_passed", False): raise RuntimeError("Gate 12 no aprobada")
print("Shrinkage values:", TAU_VALUES)
print("Salida:", OUT)


In [ ]:
def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)

def ensure_archive(runtime_name, drive_name, url, label):
    runtime, cached = RAW / runtime_name, DRIVE_BASE / drive_name
    if valid_zip(runtime): return runtime, "runtime_cache"
    if valid_zip(cached):
        shutil.copy2(cached, runtime); return runtime, "google_drive_cache"
    from google.colab import files
    print(f"Descargue y seleccione el ZIP de {label}:\n{url}")
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Suba exactamente un ZIP")
    name, content = next(iter(uploaded.items()))
    temporary = runtime.with_suffix(".zip.upload"); temporary.write_bytes(content); del content
    if not valid_zip(temporary): temporary.unlink(missing_ok=True); raise ValueError(f"{name} no es ZIP válido")
    temporary.replace(runtime); DRIVE_BASE.mkdir(parents=True, exist_ok=True); shutil.copy2(runtime, cached)
    return runtime, "manual_upload"

def safe_extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True); queue = [(Path(zip_path), destination)]; visited = set()
    while queue:
        current, target = queue.pop(0); signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited: continue
        visited.add(signature); root = target.resolve()
        with zipfile.ZipFile(current) as zipped:
            for member in zipped.infolist():
                resolved = (target / member.filename).resolve()
                if root != resolved and root not in resolved.parents: raise ValueError(f"Ruta insegura: {member.filename}")
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            if (str(nested.resolve()), str(nested_target.resolve())) not in visited: queue.append((nested, nested_target))

survey_archive, survey_method = ensure_archive("processed_edoh.zip", "processed_edoh.zip", SOURCE_URL, "microdatos")
zoning_archive, zoning_method = ensure_archive("zoning_edoh.zip", "zoning_edoh.zip", ZONING_URL, "zonificación")
safe_extract_recursive(survey_archive, INTERIM / "processed_edoh")
safe_extract_recursive(zoning_archive, INTERIM / "zoning_edoh")
print("Microdatos:", survey_method, "| Zonificación:", zoning_method)


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(c for c in value if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", value.lower().strip()).strip("_")

def clean_key(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return cleaned.mask(cleaned.map(normalize).isin(["", "nan", "none", "na", "no_aplica", "sin_informacion"]))

def canonical_utam(series):
    cleaned = clean_key(series)
    def one(value):
        if pd.isna(value): return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        match = re.fullmatch(r"UPR0*(\d+)", text)
        if match: return f"UPR{int(match.group(1)):04d}"
        match = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        return f"UTAM{int(match.group(1)):03d}" if match else text
    return cleaned.map(one).astype("string")

def parse_number(series):
    raw = series.astype("string").str.strip()
    if float(raw.str.contains(",", regex=False, na=False).mean()) > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")

def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle: sample = handle.read(32768)
            return encoding, csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
        except (UnicodeDecodeError, csv.Error): pass
    raise ValueError(f"Formato no detectado: {path}")

def find_module(token, excluded=()):
    candidates = [p for p in (INTERIM / "processed_edoh").rglob("*.csv")
                  if token in normalize(p.stem) and not any(x in normalize(p.stem) for x in excluded)]
    if not candidates: raise FileNotFoundError(token)
    return sorted(candidates, key=lambda p: (len(p.parts), len(p.name)))[0]

def read_selected(path, required):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(c): c for c in header.columns}; absent = sorted(set(required) - set(mapping))
    if absent: raise KeyError(f"Faltan columnas {absent}")
    frame = pd.read_csv(path, sep=delimiter, encoding=encoding,
                        usecols=[mapping[c] for c in required], dtype="string", low_memory=False)
    frame.columns = [normalize(c) for c in frame.columns]; return frame

trips = read_selected(find_module("viaje", excluded=("etapa",)),
                      ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo"])
persons = read_selected(find_module("persona"),
                        ["key_hg", "cod_utam_hg", "fexp_per_5anos", "sexo"])
trips["household"] = clean_key(trips["key_hg"]); trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"]); trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
persons["household"] = clean_key(persons["key_hg"]); persons["residence_utam"] = canonical_utam(persons["cod_utam_hg"])
persons["weight"] = parse_number(persons["fexp_per_5anos"]); persons["sex_group"] = persons["sexo"].map(normalize)
trips = trips[trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1) & trips["weight"].gt(0)].reset_index(drop=True)
persons = persons[persons[["household", "residence_utam"]].notna().all(axis=1) & persons["weight"].gt(0)].reset_index(drop=True)

spatial_paths = list((INTERIM / "zoning_edoh").rglob("*.shp")) + list((INTERIM / "zoning_edoh").rglob("*.gpkg"))
candidates = []
for path in spatial_paths:
    try:
        cols = {normalize(c): c for c in gpd.read_file(path, rows=5).columns}
        if "utam" in cols: candidates.append(path)
    except Exception: pass
if not candidates: raise FileNotFoundError("Capa UTAM")
utam_path = sorted(candidates, key=lambda p: ("utam2023" not in normalize(p.stem), len(p.parts)))[0]
layer = gpd.read_file(utam_path); cmap = {normalize(c): c for c in layer.columns}
mun_key = "munnombre" if "munnombre" in cmap else next(k for k in cmap if "mun" in k and "nombre" in k)
lookup = pd.DataFrame({"utam": canonical_utam(layer[cmap["utam"]]), "municipality": layer[cmap[mun_key]].map(normalize)}).dropna()
city_all_nodes = set(lookup.loc[lookup["municipality"].str.contains("bogota", na=False), "utam"])
print("Viajes:", len(trips), "| Personas:", len(persons), "| UTAM Bogotá:", len(city_all_nodes))


In [ ]:
all_households = pd.Index(sorted(set(trips["household"].astype(str)) | set(persons["household"].astype(str))))
household_to_code = {h: i for i, h in enumerate(all_households)}

def build_scope(name, scope_trips, scope_persons, nodes, P0, target_path):
    nodes = sorted(nodes); node_to_index = {node: i for i, node in enumerate(nodes)}; n = len(nodes)
    scope_trips = scope_trips[scope_trips["utam_ori"].isin(nodes) & scope_trips["utam_des"].isin(nodes)].reset_index(drop=True)
    scope_persons = scope_persons[scope_persons["residence_utam"].isin(nodes)].reset_index(drop=True)
    target_table = pd.read_csv(target_path, dtype={"utam": "string"})
    target_table["utam"] = canonical_utam(target_table["utam"])
    targets = {}
    for purpose in PURPOSES:
        mask = target_table["purpose_group"].eq(purpose)
        if "target_definition" in target_table: mask &= target_table["target_definition"].eq("core_50")
        codes = set(target_table.loc[mask, "utam"])
        missing = codes - set(nodes)
        if not codes or missing: raise RuntimeError(f"Objetivos inválidos {name}/{purpose}: {missing}")
        targets[purpose] = np.array(sorted(node_to_index[x] for x in codes), dtype=int)
    return {
        "name": name, "nodes": nodes, "n": n, "P0": P0, "targets": targets,
        "trips": scope_trips, "persons": scope_persons,
        "origin": scope_trips["utam_ori"].map(node_to_index).to_numpy(dtype=int),
        "destination": scope_trips["utam_des"].map(node_to_index).to_numpy(dtype=int),
        "trip_weight": scope_trips["weight"].to_numpy(dtype=float),
        "residence": scope_persons["residence_utam"].map(node_to_index).to_numpy(dtype=int),
        "person_weight": scope_persons["weight"].to_numpy(dtype=float),
        "trip_hh": scope_trips["household"].astype(str).map(household_to_code).to_numpy(dtype=int),
        "person_hh": scope_persons["household"].astype(str).map(household_to_code).to_numpy(dtype=int),
        "trip_masks": {g: scope_trips["sex_group"].eq(g).fillna(False).to_numpy(dtype=bool) for g in ("mujer", "hombre")},
        "person_masks": {g: scope_persons["sex_group"].eq(g).fillna(False).to_numpy(dtype=bool) for g in ("mujer", "hombre")},
    }

edges = pd.read_parquet(paths["region_edges"]); edges["utam_ori"] = canonical_utam(edges["utam_ori"]); edges["utam_des"] = canonical_utam(edges["utam_des"])
region_nodes = sorted(set(edges["utam_ori"]) | set(edges["utam_des"])); rmap = {node: i for i, node in enumerate(region_nodes)}
region_P0 = np.zeros((len(region_nodes), len(region_nodes)))
for row in edges.itertuples(index=False): region_P0[rmap[row.utam_ori], rmap[row.utam_des]] = float(row.transition_probability)

city_trips0 = trips[trips["utam_ori"].isin(city_all_nodes) & trips["utam_des"].isin(city_all_nodes)].copy()
graph = nx.DiGraph(); grouped = city_trips0.groupby(["utam_ori", "utam_des"], observed=True)["weight"].sum()
graph.add_weighted_edges_from((o, d, float(w)) for (o, d), w in grouped.items())
city_nodes = sorted(max(nx.strongly_connected_components(graph), key=len)); cmap2 = {node: i for i, node in enumerate(city_nodes)}
city_trips0 = city_trips0[city_trips0["utam_ori"].isin(city_nodes) & city_trips0["utam_des"].isin(city_nodes)].copy()
W = np.zeros((len(city_nodes), len(city_nodes)))
np.add.at(W, (city_trips0["utam_ori"].map(cmap2).to_numpy(dtype=int), city_trips0["utam_des"].map(cmap2).to_numpy(dtype=int)), city_trips0["weight"].to_numpy(dtype=float))
city_P0 = W / W.sum(axis=1)[:, None]

scopes = {
    "bogota_region": build_scope("bogota_region", trips, persons, region_nodes, region_P0, paths["region_targets"]),
    "bogota_dc_internal": build_scope("bogota_dc_internal", city_trips0, persons, city_nodes, city_P0, paths["city_targets"]),
}
display(pd.DataFrame([{ "scope": name, "nodes": s["n"], "trips": len(s["trips"]), "persons": len(s["persons"])} for name, s in scopes.items()]))


## Re-estimation across shrinkage values

For every value of $\tau$, population kernels and target sets are unchanged, while both sex-specific kernels are reconstructed with the corresponding effective-sample-size shrinkage weights.


In [ ]:
def regularized_kernel(s, level, tau):
    mask = s["trip_masks"][level]
    selected = np.flatnonzero(mask)
    original = s["trip_weight"][selected]
    origins = s["origin"][selected]
    destinations = s["destination"][selected]
    positive = original > 0

    W = np.zeros((s["n"], s["n"]))
    np.add.at(
        W,
        (origins[positive], destinations[positive]),
        original[positive],
    )
    out = W.sum(axis=1)
    sum_sq = np.bincount(
        origins[positive],
        weights=original[positive] ** 2,
        minlength=s["n"],
    )
    ess = np.divide(out ** 2, sum_sq, out=np.zeros(s["n"]), where=sum_sq > 0)
    empirical = np.zeros_like(W)
    observed = out > 0
    empirical[observed] = W[observed] / out[observed, None]
    omega = ess / (ess + float(tau))
    if float(tau) == 0:
        omega = observed.astype(float)
    return omega[:, None] * empirical + (1.0 - omega[:, None]) * s["P0"]


def residential_distribution(s, level):
    mask = s["person_masks"][level]
    selected = np.flatnonzero(mask)
    mu = np.bincount(
        s["residence"][selected],
        weights=s["person_weight"][selected],
        minlength=s["n"],
    ).astype(float)
    if not np.isfinite(mu).all() or mu.sum() <= 0:
        raise RuntimeError("Empty residential distribution")
    return mu / mu.sum()


def first_passage(s, P, targets):
    target_mask = np.zeros(s["n"], dtype=bool)
    target_mask[targets] = True
    transient = np.flatnonzero(~target_mask)
    h = np.zeros(s["n"])
    h[transient] = np.linalg.solve(
        np.eye(len(transient)) - P[np.ix_(transient, transient)],
        np.ones(len(transient)),
    )
    return h


def condition(mu, targets):
    result = mu.copy()
    result[targets] = 0
    if result.sum() <= 0:
        raise RuntimeError("Empty residential mass outside the target")
    return result / result.sum()


def gap_values(h_w, h_m, mu_w, mu_m, targets):
    mu_w = condition(mu_w, targets)
    mu_m = condition(mu_m, targets)
    mm = float(mu_m @ h_m)
    mw = float(mu_w @ h_m)
    wm = float(mu_m @ h_w)
    ww = float(mu_w @ h_w)
    residence = 0.5 * ((mw - mm) + (ww - wm))
    kernel = 0.5 * ((wm - mm) + (ww - mw))
    total = ww - mm
    return {
        "male_total_fpt": mm,
        "female_total_fpt": ww,
        "female_minus_male_total": total,
        "residential_contribution": residence,
        "kernel_contribution": kernel,
        "closure_error": total - residence - kernel,
    }


def evaluate_scope(s, tau):
    kernels = {
        level: regularized_kernel(s, level, tau)
        for level in ("mujer", "hombre")
    }
    mus = {
        level: residential_distribution(s, level)
        for level in ("mujer", "hombre")
    }
    result = {}
    for purpose in PURPOSES:
        targets = s["targets"][purpose]
        result[purpose] = gap_values(
            first_passage(s, kernels["mujer"], targets),
            first_passage(s, kernels["hombre"], targets),
            mus["mujer"],
            mus["hombre"],
            targets,
        )
    return result


point_rows = []
for tau in TAU_VALUES:
    for scope_name, scope in scopes.items():
        values = evaluate_scope(scope, tau)
        for purpose in PURPOSES:
            point_rows.append({
                "tau": tau,
                "scope": scope_name,
                "purpose_group": purpose,
                **values[purpose],
            })
    print(f"Completed tau={tau:g}")

tau_points = pd.DataFrame(point_rows)
tau_points["two_group_mean_fpt"] = (
    tau_points["female_total_fpt"] + tau_points["male_total_fpt"]
) / 2.0
tau_points["relative_gap"] = (
    tau_points["female_minus_male_total"] / tau_points["two_group_mean_fpt"]
)
tau_points["relative_gap_percent"] = 100.0 * tau_points["relative_gap"]
tau_points.to_csv(OUT / "tau_scope_point_estimates.csv", index=False)
display(tau_points.round(4))


## Paired scope contrasts

The primary sensitivity quantity is the regional-minus-district change in the absolute gap. Relative-gap contrasts are included as descriptive scale checks.


In [ ]:
contrast_rows = []
for tau in TAU_VALUES:
    frame = tau_points[tau_points["tau"].eq(tau)]
    wide = frame.pivot(index="purpose_group", columns="scope")
    for purpose in PURPOSES:
        row = {"tau": tau, "purpose_group": purpose}
        for effect in EFFECTS:
            row[f"region_minus_city__{effect}"] = float(
                wide.loc[purpose, (effect, "bogota_region")]
                - wide.loc[purpose, (effect, "bogota_dc_internal")]
            )
        city_relative = float(
            wide.loc[purpose, ("relative_gap", "bogota_dc_internal")]
        )
        region_relative = float(
            wide.loc[purpose, ("relative_gap", "bogota_region")]
        )
        row["district_relative_gap"] = city_relative
        row["regional_relative_gap"] = region_relative
        row["region_minus_city__relative_gap"] = region_relative - city_relative
        contrast_rows.append(row)

tau_contrasts = pd.DataFrame(contrast_rows)
tau_contrasts.to_csv(OUT / "tau_paired_scope_contrasts.csv", index=False)
display(tau_contrasts.round(4))

summary = tau_contrasts.groupby("purpose_group", observed=True).agg(
    minimum_scope_amplification=("region_minus_city__female_minus_male_total", "min"),
    maximum_scope_amplification=("region_minus_city__female_minus_male_total", "max"),
    minimum_relative_amplification=("region_minus_city__relative_gap", "min"),
    maximum_relative_amplification=("region_minus_city__relative_gap", "max"),
).reset_index()
summary.to_csv(OUT / "tau_sensitivity_ranges.csv", index=False)
display(summary.round(4))

sns.set_theme(style="whitegrid", context="notebook")
fig, axis = plt.subplots(figsize=(8.8, 5.4))
labels = {"employment": "Employment", "education": "Education", "health": "Health"}
colors = {"employment": "#355C9A", "education": "#2A9D8F", "health": "#D46A3A"}
for purpose in PURPOSES:
    frame = tau_contrasts[tau_contrasts["purpose_group"].eq(purpose)].sort_values("tau")
    axis.plot(
        frame["tau"],
        frame["region_minus_city__female_minus_male_total"],
        marker="o",
        linewidth=2,
        label=labels[purpose],
        color=colors[purpose],
    )
axis.axhline(0, color="black", linewidth=0.8)
axis.axvline(PRIMARY_TAU, color="#666666", linestyle="--", linewidth=1.0)
axis.set(
    xlabel="Shrinkage parameter tau",
    ylabel="Bogotá–Region minus Bogotá D.C. gap (FPT steps)",
)
axis.legend(title="Destination purpose", frameon=False)
fig.tight_layout()
fig.savefig(
    OUT / "figure_tau_scope_amplification_sensitivity.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Validation against the primary specification and persistent outputs


In [ ]:
validated = pd.concat([
    pd.read_csv(paths["region_point"]).assign(scope="bogota_region"),
    pd.read_csv(paths["city_point"]).assign(scope="bogota_dc_internal"),
], ignore_index=True)

primary = tau_points[tau_points["tau"].eq(PRIMARY_TAU)]
comparison_columns = [
    "male_total_fpt",
    "female_total_fpt",
    "female_minus_male_total",
    "residential_contribution",
    "kernel_contribution",
]
merged = primary.merge(
    validated,
    on=["scope", "purpose_group"],
    suffixes=("_new", "_validated"),
)
replication_errors = {}
for column in comparison_columns:
    replication_errors[column] = float(
        (merged[f"{column}_new"] - merged[f"{column}_validated"]).abs().max()
    )
maximum_replication_error = max(replication_errors.values())

technical_checks = {
    "gate11_dependency_valid": True,
    "gate12_dependency_valid": True,
    "all_tau_values_complete": bool(
        len(tau_points) == len(TAU_VALUES) * 2 * len(PURPOSES)
    ),
    "all_scope_purpose_tau_rows_unique": bool(
        tau_points.groupby(
            ["tau", "scope", "purpose_group"], observed=True
        ).size().eq(1).all()
    ),
    "all_outputs_finite": bool(
        np.isfinite(tau_points.select_dtypes(include="number")).all().all()
        and np.isfinite(tau_contrasts.select_dtypes(include="number")).all().all()
    ),
    "all_kernels_closed_algebraically": bool(
        tau_points["closure_error"].abs().max() <= 1e-10
    ),
    "primary_tau_reproduces_validated_results": bool(
        maximum_replication_error <= 1e-9
    ),
}
technical_checks["gate_20_passed"] = bool(all(technical_checks.values()))

diagnostics = {
    "all_absolute_scope_contrasts_positive_across_tau": bool(
        tau_contrasts[
            "region_minus_city__female_minus_male_total"
        ].gt(0).all()
    ),
    "all_relative_scope_contrasts_positive_across_tau": bool(
        tau_contrasts["region_minus_city__relative_gap"].gt(0).all()
    ),
}
report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "point-estimate sensitivity of scope gaps to row shrinkage",
    "tau_values": list(TAU_VALUES),
    "primary_tau": PRIMARY_TAU,
    "inference_boundary": (
        "This gate reports point-estimate sensitivity. Simultaneous inference "
        "remains attached to the primary tau=20 specification."
    ),
    "maximum_primary_replication_error": maximum_replication_error,
    "replication_errors": replication_errors,
    "diagnostics": diagnostics,
    "technical_gate": technical_checks,
    "gate": {"gate_20_passed": technical_checks["gate_20_passed"]},
}
(OUT / "gate_20.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series({**technical_checks, **diagnostics}, name="result").to_csv(
    OUT / "gate_20_checks.csv"
)
display(pd.Series({**technical_checks, **diagnostics}, name="result"))
print("Maximum tau=20 replication error:", maximum_replication_error)

if not technical_checks["gate_20_passed"]:
    raise RuntimeError("Gate 20 failed; inspect gate_20_checks.csv")

persistent = DRIVE_BASE / "results/gate20"
persistent.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent / artifact.name)

print("Gate 20: APPROVED")
print("Persistent copy:", persistent)
